# M1: LivePortrait — Talking Head Generation

## Overview

**LivePortrait** is a portrait-animation model developed by KwaiVGI that transfers facial motion from a *driving video* onto a *static source image*, preserving the identity of the source portrait throughout.

### How it works in this pipeline

1. **Source image** — one of our 8 static headshot PNGs (`person_01.png` … `person_08.png`).
2. **Driving (template) video** — a short (~5–10 s) generic talking-head clip uploaded once to Google Drive. This clip provides *motion only* (lip sync, eye blinks, head micro-movements). The identity of the person in the template is **not** transferred — only their movement.
3. **Output** — LivePortrait renders a new video where the source headshot appears to speak with the same motion as the template, at 25 fps.

We loop the template video over all 8 headshots in a single batch run, saving each result as `person_NN_talking.mp4` in Google Drive.

### Runtime requirements
- Google Colab **GPU runtime** (T4 or better — free tier is sufficient)
- Make sure `Runtime → Change runtime type → Hardware accelerator → GPU` is selected before running any cell.

In [ ]:
# ── Cell 2: Install LivePortrait + dependencies (Colab only) ─────────────────
# When running locally, dependencies are managed via the project .venv.
# This cell is only needed in Google Colab.
import os, subprocess, sys

def _run(cmd):
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

# 1. Clone repo
if not os.path.isdir('/content/LivePortrait'):
    print('Cloning LivePortrait repo...')
    _run('git clone https://github.com/KwaiVGI/LivePortrait /content/LivePortrait')
else:
    print('Repo already present.')

# 2. PyTorch (Colab already has CUDA PyTorch — skip if available)
import importlib
if importlib.util.find_spec('torch') is None:
    print('Installing PyTorch (CPU)...')
    _run('pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu')

# 3. Base requirements only (avoids onnxruntime-gpu which needs CUDA)
print('Installing base requirements...')
_run('pip install -q -r /content/LivePortrait/requirements_base.txt')

# 4. CPU onnxruntime + extras
print('Installing onnxruntime + huggingface_hub...')
_run('pip install -q onnxruntime transformers==4.38.0 huggingface_hub')

# 5. Download model weights from HuggingFace KlingTeam/LivePortrait
REPO_DIR = '/content/LivePortrait'
WEIGHTS_LOCAL_DIR = f'{REPO_DIR}/pretrained_weights'
os.makedirs(WEIGHTS_LOCAL_DIR, exist_ok=True)

from huggingface_hub import hf_hub_download
WEIGHT_FILES = [
    'liveportrait/base_models/appearance_feature_extractor.pth',
    'liveportrait/base_models/motion_extractor.pth',
    'liveportrait/base_models/spade_generator.pth',
    'liveportrait/base_models/warping_module.pth',
    'liveportrait/retargeting_models/stitching_retargeting_module.pth',
    'liveportrait/landmark.onnx',
]
print('Downloading model weights from HuggingFace...')
for f in WEIGHT_FILES:
    dest = f'{WEIGHTS_LOCAL_DIR}/{f}'
    if os.path.exists(dest):
        print(f'  Already exists: {f}')
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f'  Downloading {os.path.basename(f)} ...')
    hf_hub_download(repo_id='KlingTeam/LivePortrait', filename=f, local_dir=WEIGHTS_LOCAL_DIR)

print('Setup complete.')

In [ ]:
# ── Cell 2b: Generate the hybrid driving template (Colab only) ───────────────
# Combines talking.pkl's high head/shoulder rotation with d8.pkl's lip expression variety.
# On CPU this is instant (pure numpy, no GPU needed).
import pickle, numpy as np, os
from pathlib import Path

DRIVE_DIR_COLAB = '/content/LivePortrait/assets/examples/driving'
out_path = f'{DRIVE_DIR_COLAB}/talking_with_movement.pkl'

if not os.path.exists(out_path):
    with open(f'{DRIVE_DIR_COLAB}/talking.pkl', 'rb') as f:
        talking = pickle.load(f)
    with open(f'{DRIVE_DIR_COLAB}/d8.pkl', 'rb') as f:
        d8 = pickle.load(f)

    n = talking['n_frames']
    d8_motion = d8['motion']
    indices = np.linspace(0, len(d8_motion)-1, n).astype(int)
    d8_exps = np.stack([d8_motion[i]['exp'] for i in indices])

    t_arr = np.linspace(0, 4*np.pi, n)
    blink = np.where(np.abs(np.sin(t_arr*0.7)) > 0.97, 0.05, 0.26).astype(np.float32)
    lip   = ((np.sin(t_arr)*0.4 + 0.5) * np.abs(np.sin(t_arr*2.3))) * 0.18

    hybrid = {
        'n_frames': n,
        'output_fps': talking['output_fps'],
        'motion': [
            {'scale': talking['motion'][i]['scale'],
             'R_d':   talking['motion'][i]['R'],
             'exp':   d8_exps[i],
             't':     talking['motion'][i]['t']}
            for i in range(n)
        ],
        'c_d_eyes_lst': [np.array([[b, b]], dtype=np.float32) for b in blink],
        'c_d_lip_lst':  [np.array([[float(lip[i])]], dtype=np.float32) for i in range(n)],
    }
    with open(out_path, 'wb') as f:
        pickle.dump(hybrid, f)
    print(f'Generated hybrid template -> {out_path}')
else:
    print(f'Hybrid template already exists: {out_path}')

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    DRIVE_BASE    = '/content/drive/MyDrive/talking_head'
    HEADSHOTS_DIR = f'{DRIVE_BASE}/headshots'
    OUTPUT_DIR    = f'{DRIVE_BASE}/outputs/m1/liveportrait'
    REPO_DIR      = '/content/LivePortrait'
    # Hybrid template: 10× head/shoulder motion from talking.pkl + d8.pkl lip variety
    DEFAULT_DRIVING = f'{REPO_DIR}/assets/examples/driving/talking_with_movement.pkl'
    print("Running in Google Colab")
except ImportError:
    import sys
    IN_COLAB = False
    PROJECT_ROOT = Path().resolve()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))
    HEADSHOTS_DIR = str(PROJECT_ROOT / 'headshots')
    OUTPUT_DIR    = str(PROJECT_ROOT / 'outputs' / 'm1' / 'liveportrait')
    DEFAULT_DRIVING = ''  # scripts/m1/liveportrait.py resolves the hybrid template internally
    print(f"Running locally. Project root: {PROJECT_ROOT}")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Headshots  : {HEADSHOTS_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

In [ ]:
import glob, os, sys, subprocess
from pathlib import Path

if IN_COLAB:
    # ── Inline inference for Colab ─────────────────────────────────────────
    def generate_animation(headshot_path, output_dir, template_video='', fps=25, duration_sec=5.0):
        """Run LivePortrait inference on a single headshot."""
        headshot_path = str(Path(headshot_path).resolve())
        output_dir = str(Path(output_dir).resolve())
        driving = template_video if template_video else DEFAULT_DRIVING
        os.makedirs(output_dir, exist_ok=True)
        name = Path(headshot_path).stem
        lp_output_dir = os.path.join(output_dir, name)
        os.makedirs(lp_output_dir, exist_ok=True)

        cmd = [
            sys.executable, f'{REPO_DIR}/inference.py',
            '--source', headshot_path,
            '--driving', driving,
            '--output_dir', lp_output_dir,
        ]
        # On GPU (Colab): keep FP16 and don't force CPU
        env = os.environ.copy()
        env['PYTHONUTF8'] = '1'
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR, env=env)
        if result.returncode != 0:
            raise RuntimeError(f"LivePortrait failed:\n{result.stderr[-2000:]}")

        # Find and move output to standardized name
        candidates = sorted(
            glob.glob(os.path.join(lp_output_dir, '**', '*.mp4'), recursive=True),
            key=os.path.getmtime, reverse=True,
        )
        if not candidates:
            raise RuntimeError(f"No output mp4 in {lp_output_dir}")
        output_path = os.path.join(output_dir, f'{name}_talking.mp4')
        os.replace(candidates[0], output_path)
        return {'output_path': output_path, 'duration': duration_sec,
                'engine': 'liveportrait', 'fps': fps,
                'metadata': {'driving': driving, 'source': headshot_path}}
else:
    # ── Local import (no code duplication) ──────────────────────────────────
    from scripts.m1.liveportrait import generate_animation
    print("generate_animation imported from scripts/m1/liveportrait.py")

# ── Batch: run all headshots ─────────────────────────────────────────────────
headshots = sorted(glob.glob(os.path.join(HEADSHOTS_DIR, 'person_*.png')))
print(f"Found {len(headshots)} headshot(s)")

results, failures = [], []
for hs in headshots:
    name = Path(hs).stem
    print(f"Processing {name} ...", end=" ", flush=True)
    try:
        r = generate_animation(hs, OUTPUT_DIR, template_video=DEFAULT_DRIVING)
        results.append(r)
        size_kb = os.path.getsize(r['output_path']) // 1024
        print(f"Done ({size_kb} KB)")
    except Exception as e:
        failures.append({"headshot": hs, "error": str(e)})
        print(f"FAILED: {e}")

print(f"\nCompleted {len(results)}/{len(headshots)}  |  Failed: {len(failures)}")

## Quality Check

Review each of the 8 output videos against the checklist below before signing off on Milestone 1.

### Per-video checklist

| # | Check | person_01 | person_02 | person_03 | person_04 | person_05 | person_06 | person_07 | person_08 |
|---|-------|-----------|-----------|-----------|-----------|-----------|-----------|-----------|-----------|
| 1 | Mouth movement looks natural | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| 2 | Eyes blink naturally | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| 3 | No severe warping or structural distortion | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| 4 | Loop point is seamless (end → start transition) | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| 5 | Head/shoulder micro-movements present | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |

### How to verify the loop

- **VLC** → `Media → Open File` → right-click the video in the playlist → `Loop`. Watch the cut between the last frame and the first frame — it should feel continuous, not jarring.
- **Browser** → drag the `.mp4` file into a Chrome or Firefox tab. The browser will auto-loop the video (it loops by default when the `loop` attribute is implicit via the media player). Watch the repeat a few times.
- **Quick visual test** → scrub to the last 10 frames in any video player and compare mouth/head position with the first 10 frames.

### Common issues and fixes

| Issue | Likely cause | Fix |
|-------|-------------|-----|
| Severe warping / skin deformation | Source headshot has extreme lighting or angle | Re-crop headshot to a straight-on, well-lit close-up |
| Mouth barely moves | Template video is too quiet / minimal mouth movement | Replace template with a clip of more exaggerated speech |
| Ghosting artefacts | GPU memory issue during inference | Restart runtime, run one image at a time |
| Output file not found after inference | LivePortrait wrote to a subdirectory | Check `/content/LivePortrait/results/` and copy manually |
| `FileNotFoundError: template_talking.mp4` | Template not yet uploaded to Drive | Upload the file and re-run Cell 4 only |